In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os

In [ ]:
#Paths
IMG_DIR="/content/drive/MyDrive/CheXchoNet/images"
CSV_PATH="/content/drive/MyDrive/CheXchoNet/metadata.csv"

#Model config
IMG_SIZE=260
BATCH_SIZE=8
EPOCHS=10

In [ ]:
#Load csv data and selecting label
df=pd.read_csv(CSV_PATH)
df = df[['cxr_filename', 'slvh', 'dlv']]
df = df.rename(columns={'cxr_filename': 'image_id'})

print(df.head())
print("Total samples:", len(df))

                                        image_id  slvh  dlv
0  0001a10b30d6cff4ce1c2bff5be86958_6aee7800.jpg     0    0
1  0008eb0430637bd17906cb0a5f1126e1_000bd69b.jpg     0    0
2  0008eb0430637bd17906cb0a5f1126e1_301ec85b.jpg     0    0
3  000a92e15c0a8df9f107f09acf295e36_32b2a209.jpg     0    0
4  00163791112923cb52834f0e75123225_32f21836.jpg     0    0
Total samples: 71589


In [ ]:
# Initial random 10k
df_base = df.sample(n=7000, random_state=42)

# Remaining data (unused)
df_remaining = df.drop(df_base.index)

In [ ]:
# Prefer SLVH or DLV cases from remaining data
df_disease = df_remaining[
    (df_remaining['slvh'] == 1) | (df_remaining['dlv'] == 1)
]

# Take 5000 more (increase later to 10k / 20k)
df_extra = df_disease.sample(n=3000, random_state=42)


In [ ]:
df_small = pd.concat([df_base, df_extra]).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_small,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("SLVH positives:", df_small['slvh'].sum())
print("DLV positives:", df_small['dlv'].sum())
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

SLVH positives: 2448
DLV positives: 1761
Train: 8000
Val: 1000
Test: 1000


In [ ]:
def load_image(filename, label):
    img_path=tf.strings.join([IMG_DIR, '/', filename])
    img=tf.io.read_file(img_path)
    img=tf.image.decode_jpeg(img, channels=3)
    img=tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img=img / 255.0
    return img, label

In [ ]:
#Concating image with label
def make_dataset(df,training=True):
  labels=df[['slvh','dlv']].values.astype('float32')
  ds=tf.data.Dataset.from_tensor_slices(
      (df['image_id'].values,labels)
  )
  ds=ds.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)

  if training:
    ds=ds.shuffle(1000)

  ds=ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
  return ds

In [ ]:
train_ds=make_dataset(train_df,training=True)
val_ds=make_dataset(val_df,training=False)
test_ds=make_dataset(test_df,training=False)

In [ ]:
print(train_ds)
print(val_ds)
print(test_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2), dtype=tf.float32, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 2), dtype=tf.float32, name=None))>


In [ ]:
#EfficientNet-B2 Base Model
base_model=tf.keras.applications.EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

base_model.trainable=False

In [ ]:
model=tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2,activation='sigmoid')
])

In [ ]:
#Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Recall(thresholds=0.3, name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)


In [ ]:
CKPT_DIR = "/content/drive/MyDrive/CheXchoNet/checkpoints4"
os.makedirs(CKPT_DIR, exist_ok=True)


In [ ]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=CKPT_DIR + "/epoch_{epoch:02d}.keras",
    save_weights_only=False,   # saves full model
    save_freq="epoch",
    verbose=1
)


In [ ]:
both = ((df_small["slvh"] == 1) & (df_small["dlv"] == 1)).sum()
only_slvh = ((df_small["slvh"] == 1) & (df_small["dlv"] == 0)).sum()
only_dlv  = ((df_small["slvh"] == 0) & (df_small["dlv"] == 1)).sum()
normal    = ((df_small["slvh"] == 0) & (df_small["dlv"] == 0)).sum()

print("Normal:", normal)
print("Only SLVH:", only_slvh)
print("Only DLV:", only_dlv)
print("Both:", both)


Normal: 6061
Only SLVH: 2178
Only DLV: 1491
Both: 270


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb]
)


Epoch 1/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7955 - auc: 0.5416 - loss: 0.5081 - recall: 0.0684
Epoch 1: saving model to /content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_01.keras
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1370s 1s/step - accuracy: 0.7955 - auc: 0.5416 - loss: 0.5081 - recall: 0.0684 - val_accuracy: 0.7860 - val_auc: 0.5699 - val_loss: 0.5181 - val_recall: 0.0000e+00
Epoch 2/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7951 - auc: 0.5427 - loss: 0.5074 - recall: 0.0610
Epoch 2: saving model to /content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_02.keras
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1338s 1s/step - accuracy: 0.7951 - auc: 0.5427 - loss: 0.5074 - recall: 0.0611 - val_accuracy: 0.7860 - val_auc: 0.5609 - val_loss: 0.5157 - val_recall: 0.0000e+00
Epoch 3/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7938 - auc: 0.5343 - loss: 0.5104 - recall: 0.0384
Epoch 3: saving model to /content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_03.k

In [ ]:
val_accuracy = history.history['val_accuracy'][-1]
print(f"Final Validation Accuracy: {val_accuracy:.4f}")

Final Validation Accuracy: 0.7860


In [ ]:
import tensorflow as tf

MODEL_IN = "/content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_10.keras"
WEIGHTS_OUT = "/content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_10.weights.h5"

m = tf.keras.models.load_model(MODEL_IN, compile=False)
m.save_weights(WEIGHTS_OUT)
print("Saved:", WEIGHTS_OUT)


Saved: /content/drive/MyDrive/CheXchoNet/checkpoints4/epoch_10.weights.h5
